# MACURA on a PyBullet quadrotor — study notebook

Compare four RL algorithms — **MACURA, MBPO, M2AC, SAC** — on an unstable
quadrotor hover task. Algorithm logic lives in the `.py` modules; this notebook
only imports, runs, and plots. Everything saves to Drive. Set **`SMOKE=True`**
for a quick pass; `False` for the ~20k-step run.

## 2. Imports & setup
Mount Drive, require a GPU, clone+install, print versions, load the config.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
os.environ['MUJOCO_GL'] = 'egl'
warnings.filterwarnings('ignore')
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
os.environ['GITHUB_TOKEN'] = TOKEN
print('token loaded')

In [ ]:
%%bash
set -e
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/silvergjeka22/macura-drone/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys
if '/content/macura-drone' not in sys.path:
    sys.path.insert(0, '/content/macura-drone')
import copy, json, yaml, numpy as np
import pybullet, gymnasium, stable_baselines3
PYB_VER = getattr(pybullet, '__version__', None) or pybullet.getAPIVersion()
print('torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| pybullet', PYB_VER, '| gymnasium', gymnasium.__version__, '| numpy', np.__version__)

from envs import drone_env
from viz import plots
from training import train as trainer
from algorithms import sac as sac_mod
from models import ensemble as ens
from IPython.display import Video, display
import pandas as pd

CFG = yaml.safe_load(open('/content/macura-drone/configs/macura_drone.yaml'))
DRIVE = CFG['experiment']['drive_root']
SMOKE = True   # True = tiny quick pass; False = ~20k-step run

def make_cfg(smoke=False, fast=True):
    c = copy.deepcopy(CFG)
    if fast:
        for k, v in c.get('profiles', {}).get('fast', {}).items():
            c[k].update(v) if isinstance(v, dict) else c.__setitem__(k, v)
    if smoke:
        c['experiment'].update(total_env_steps=1500, warmup_random_steps=300,
                               eval_every_steps=500, eval_episodes=2, seeds=[0])
        c['rollout'].update(num_rollouts=100, freq_steps=250)
    c['env']['backend'] = 'pybullet'
    c['env']['mjcf_scene'] = '/content/mujoco_menagerie/skydio_x2/scene.xml'
    return c

cfg = make_cfg(smoke=SMOKE, fast=True)
print('steps', cfg['experiment']['total_env_steps'], '| seeds', cfg['experiment']['seeds'])

## 3. Explore the environment
**Task:** a Crazyflie quadrotor must take off and hover at `(0, 0, 1 m)`.
Observation = 13-dim kinematic state; action = 4 rotor commands; dense quadratic
reward. **Unlike** the paper's stable MuJoCo locomotion, a quadrotor is unstable
(it balances on thrust), so model errors compound fast — a harder test for the
model-based methods.

In [ ]:
env, obs_dim, act_dim = drone_env.make_env(cfg['env'], seed=0, render=True)
env.reset(seed=0)
print('obs_dim', obs_dim, '| act_dim', act_dim, '| HOVER_RPM', round(env.HOVER_RPM, 1))

A ~12 s video of the drone (camera follows it and zooms in). Pre-training it just
tumbles and resets — it's there to show the env clearly. Saved to Drive + shown.

In [ ]:
vid = f'{DRIVE}/videos/env_demo.mp4'
plots.record_env_video(env, policy='random', seconds=12, fps=30, save_path=vid)
display(Video(vid, embed=True, width=520))

One rollout's traces (altitude, position error, tilt, reward) under random
actions — the tilt/altitude blowing up is the instability the agent must tame.

In [ ]:
rec = plots.collect_rollout(env, policy='random', num_steps=200, seed=1)
plots.plot_rollout_traces(rec, title='Random actions', save_path=f'{DRIVE}/plots/s3_random.png');

## 4. The four algorithms
All share the **same SAC backbone** and (model-based ones) the **same ensemble**;
only the rollout differs — **MACURA** adaptive uncertainty-truncated, **MBPO**
fixed length, **M2AC** masking, **SAC** none. The schematic shows rollout length
vs training step for each.

In [ ]:
agent = sac_mod.build_sac(obs_dim, act_dim, {**cfg['sac'], 'buffer_size': 10000}, 'cuda', 0)
ensemble = ens.build_ensemble(cfg['ensemble'], obs_dim, act_dim, 'cuda')
print('shared SAC params:', sum(p.numel() for p in agent.policy.parameters()),
      '| ensemble members:', len(ensemble['model'].members))
plots.plot_rollout_strategy_schematic(save_path=f'{DRIVE}/plots/s4_strategy.png');

## 5. Exploration noise
We use **pink noise** (`1/f`, temporally correlated) — smooth, momentum-like
exploration that suits continuous drone control, applied to all four for fairness
(real FFT generator in `training/train.py`). White noise is uncorrelated/jittery;
init noise is always on; sensor noise is off by default. The plot shows both
signals and their power spectra (pink has the `1/f` slope).

In [ ]:
plots.plot_noise_comparison(save_path=f'{DRIVE}/plots/s5_noise.png');

## 6. Training (fast, ~20k steps)
Train all four (fast profile; best model per algo/seed saved to Drive, periodic
eval). **`FRESH=True` retrains now**; set `False` to reuse existing Drive logs.

In [ ]:
FRESH = True   # True = (re)train now; False = reuse existing logs in Drive
runs = []
for algo in cfg['experiment']['algorithms']:
    for seed in cfg['experiment']['seeds']:
        lp = f'{DRIVE}/logs/{algo}_seed{seed}.json'
        if (not FRESH) and os.path.exists(lp):
            print('reuse:', algo, seed); runs.append(json.load(open(lp)))
        else:
            runs.append(trainer.train_one(algo, cfg, DRIVE, seed=seed))
print('done:', len(runs), 'runs')

Learning curves: evaluation return vs real environment steps (mean ± std).

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/s6_sample_eff.png');

## 7. Evaluate best models + videos
Load each best checkpoint from Drive, evaluate it, and save a ~12 s `.mp4` per
algorithm/seed; display inline.

In [ ]:
display(pd.DataFrame(trainer.evaluate_best(cfg, DRIVE, device='cuda')))
video_paths = trainer.record_best_videos(cfg, DRIVE, device='cuda', seconds=12)
for name, path in video_paths.items():
    print(name)
    display(Video(path, embed=True, width=480))

## 8. Results comparison
Final-policy quality per algorithm and the MACURA signature (rollout length &
kappa over training). With `SMOKE=True` the curves are short — run `SMOKE=False`
for the real comparison.

Read it honestly: does the model help (MACURA/MBPO/M2AC vs SAC)? does adaptivity
help (MACURA vs MBPO)? truncation vs masking (MACURA vs M2AC)?

In [ ]:
plots.plot_final_quality(runs, save_path=f'{DRIVE}/plots/s8_final_quality.png')
mac = next((r for r in runs if r['algo'] == 'macura'), None)
if mac:
    plots.plot_rollout_depth(mac, save_path=f'{DRIVE}/plots/s8_rollout_depth.png')
display(pd.DataFrame([{'algo': r['algo'], 'seed': r['seed'],
                       'best_return': round(r.get('best_return', 0.0), 1)} for r in runs]))